<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Case study: card fraud with a review budget

**[Case study: card fraud with a review budget](https://learning.nextia-ai.com/courses/anomaly-detection/m10/card-fraud-with-a-review-budget/)** · Anomaly Detection: Find What Is Unusual · Module 10, case study 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** take anonymised card payments from the first look to a daily review queue. You will compare a rule, two detectors without labels and a supervised model at a budget of 20 alerts a day, choose by the cost per week, and run the choice on a month it has never seen.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about one minute after the install |
| **Needs** | An internet connection for the setup cells (21.5 MB). No account. |
| **You should know** | The alert budget, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/); features from the customer's past, from [Features decide more than the algorithm](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/). |
| **Data** | Simulated card payments (Sparkov), CC0 1.0, by Kartik Shenoy on Kaggle. Details in `dataset.md` next to this notebook. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m10/card-fraud-with-a-review-budget/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L01-card-fraud-with-a-review-budget/card-fraud-with-a-review-budget-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the data

Run the next cell. It downloads `card_transactions.csv.gz` (21.5 MB) into a folder `anomaly-course/data` and checks its checksum. It needs an internet connection and no account.

You should see `Ready:` and the file with its size. If you run the cell again, it uses the file that is already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course`, then run the cell again.

The payments are **simulated**: no real people or cards. The source is "Credit Card Transactions Fraud Detection Dataset" by Kartik Shenoy (Kaggle), made with Brandon Harris's Sparkov generator, CC0 1.0. `dataset.md` lists what we changed.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L01-card-fraud-with-a-review-budget/"
FILES = {
    "card_transactions.csv.gz": "3ad8405b69038b3c36cb24beeaad194cbed4caee167b0f1024bc2de7703a16a0",
}
CODE_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The first cell defines the check helpers `expect` and `expect_hash`. They tell you if your answer is right, without showing it. It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`.

The second cell defines the case study's own helpers. `review()` gives each day's queue: the team reads the `BUDGET` highest scores of the day. `score_at_budget()` returns one row of the comparison table: precision at the budget, recall and cost per week. The costs are the course's **assumptions**: a missed fraud costs its amount plus a 15-dollar chargeback fee, and a false alert costs 3 dollars. Run both cells. They print nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

In [ ]:
# The case study's helpers: the review budget, the costs, and one row of the comparison table.
BUDGET = 20              # alerts a day that the review team can read (assumption)
CHARGEBACK_FEE = 15.0    # dollars added to each missed fraud (assumption, the course's number)
FALSE_ALERT_COST = 3.0   # dollars per false alert: about 4 minutes of a reviewer (assumption)

def review(part, score, budget=BUDGET, threshold=None):
    """Each day, the team reviews the `budget` highest scores (ties in time order).
    With a threshold, only scores at or above it are alerts; alerts beyond the budget wait (missed)."""
    p = part[["ts", "amount", "is_fraud"]].copy()
    p["score"] = np.asarray(score, dtype=float)
    p["day"] = p.ts.dt.floor("D")
    p["alert"] = True if threshold is None else p.score >= threshold
    p = p.sort_values(["day", "score", "ts"], ascending=[True, False, True], kind="stable")
    p["rank"] = p[p.alert].groupby("day").cumcount()
    p["reviewed"] = p.alert & (p["rank"] < budget)
    if threshold is None:
        p["alert"] = p.reviewed          # top-k: the detector sends exactly the budget
    return p.sort_index()

def score_at_budget(part, score, detector, setting="", budget=BUDGET, threshold=None):
    """One row of the comparison table: precision at the budget, recall and cost per week."""
    p = review(part, score, budget, threshold)
    days = p.day.nunique()
    weeks = days / 7
    reviewed = p[p.reviewed]
    caught = int(reviewed.is_fraud.sum())
    missed = p[(p.is_fraud == 1) & ~p.reviewed]
    false_alerts = len(reviewed) - caught
    cost = (missed.amount.sum() + CHARGEBACK_FEE * len(missed) + FALSE_ALERT_COST * false_alerts) / weeks
    return {"detector": detector, "setting": setting,
            "alerts per day": round(p.alert.sum() / days, 1),
            "precision at budget": round(caught / len(reviewed), 3) if len(reviewed) else 0.0,
            "recall": round(caught / max(int(p.is_fraud.sum()), 1), 3),
            "cost per week ($)": round(float(cost)),
            "false alerts per week": round(false_alerts / weeks, 1)}

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the first checks

One row is one card payment: the time, the card, the shop category, the amount and the label `is_fraud`.

> **Check.** You should see 1,852,394 transactions of 999 cards, from 2019-01-01 to 2020-12-31, and 9,651 frauds (0.52%).

In [ ]:
tx = pd.read_csv("data/card_transactions.csv.gz", parse_dates=["ts"])
day = tx.ts.dt.floor("D")
print(f"{len(tx):,} transactions, {tx.card_id.nunique()} cards, {day.min():%Y-%m-%d} to {day.max():%Y-%m-%d}")
print(f"fraud: {tx.is_fraud.sum():,} ({tx.is_fraud.mean():.2%}) | cards with fraud: {tx[tx.is_fraud == 1].card_id.nunique()}")
print(f"missing values: {tx.isna().sum().sum()} | repeated rows: {tx.duplicated().sum()} | amount from {tx.amount.min()} to {tx.amount.max():,}")
print("days with no payment:", [f"{d:%Y-%m-%d}" for d in pd.date_range(day.min(), day.max()).difference(day.unique())])
per_day = tx.groupby(day).agg(transactions=("is_fraud", "size"), fraud=("is_fraud", "sum"))
print(per_day.describe().loc[["mean", "min", "50%", "max"]].round(1).T.to_string())

No missing values and no repeated rows. A day has about 2,540 payments and 13 frauds. The team can read 20 alerts a day, so on many days there are fewer frauds than places, and on some days more.

Now look at how fraud happens in this data:

In [ ]:
fraud = tx[tx.is_fraud == 1]
episode = fraud.groupby("card_id").ts.agg(["min", "max", "size"])
episode["days"] = (episode["max"] - episode["min"]) / pd.Timedelta("1D")
print(f"fraud episodes: {len(episode)} | payments per episode: median {episode['size'].median():.0f} "
      f"(from {episode['size'].min()} to {episode['size'].max()}) | days per episode: median {episode.days.median():.1f}")
night = (fraud.ts.dt.hour >= 22) | (fraud.ts.dt.hour < 4)
print(f"fraud between 22:00 and 03:59: {night.mean():.1%} | all payments: {((tx.ts.dt.hour >= 22) | (tx.ts.dt.hour < 4)).mean():.1%}")
print("median amount ($):", tx.groupby("is_fraud").amount.median().rename({0: "honest", 1: "fraud"}).to_dict())

> **Check.** You should see 976 fraud episodes, about 10 payments in about 2 days each, and 84.6% of the frauds between 22:00 and 03:59.

So fraud comes in short bursts on one card, mostly late at night, with larger amounts. This is a simulator: real fraud is less tidy. Keep that in mind when you read the results.

## 2. The base-rate trap

> **Predict.** A detector finds 90% of the frauds and wrongly flags only 1% of the honest payments. Of its alerts, how many are real frauds?

In [ ]:
part = tx[(day >= "2020-01-01") & (day < "2020-07-01")]
print(f"accuracy of 'nothing is fraud': {1 - part.is_fraud.mean():.2%}")
# A detector that finds 90% of the frauds and wrongly flags only 1% of the honest payments:
frauds, honest = part.is_fraud.sum(), (part.is_fraud == 0).sum()
caught, false_alerts = 0.90 * frauds, 0.01 * honest
n_days = part.ts.dt.floor("D").nunique()
print(f"per day: {frauds / n_days:.1f} frauds, {honest / n_days:,.0f} honest payments")
print(f"alerts per day: {(caught + false_alerts) / n_days:.1f} ({caught / n_days:.1f} real, {false_alerts / n_days:.1f} false)"
      f" | precision {caught / (caught + false_alerts):.3f} | accuracy {(caught + honest - false_alerts) / len(part):.2%}")

> **Check.** You should see an accuracy of 99.40% for "nothing is fraud", and a precision of only 0.353 for the "90% and 1%" detector, with 34.1 alerts a day for a team that can read 20.

Fraud is rare, so 1% of the many honest payments is more than 90% of the few frauds. Accuracy hides this. The lesson page has the picture.

## 3. Features from the card's own past

The next cell builds the features. Each one uses only the card's payments **before** this one, as a live system would. A card with no past yet is compared with a typical card of 2019.

In [ ]:
def card_features(tx):
    """Features from each card's own past: only payments before (or at) this one. `tx` is sorted by time.
    A card with no past yet is compared with a typical card of 2019."""
    t = tx[["ts", "card_id", "amount"]]
    by_card = t.groupby("card_id", sort=False)
    hour = t.ts.dt.hour
    night = ((hour >= 22) | (hour < 6)).astype(float)                       # 22:00 to 05:59
    last_30 = lambda q: by_card.amount.transform(lambda s: s.shift().rolling(30, min_periods=5).quantile(q))
    median, q1, q3 = last_30(0.5), last_30(0.25), last_30(0.75)              # of the card's last 30 amounts

    def last_24h(values):        # the sum over the card's last 24 hours, this payment included
        return values.groupby(t.card_id).transform(
            lambda s: s.set_axis(t.ts[s.index]).rolling("24h").sum().to_numpy())

    d = t.ts.dt.floor("D")
    spend = t.groupby(["card_id", d]).amount.sum()                          # each card's spend per day
    usual_day = spend.groupby(level=0).transform(lambda s: s.shift().rolling(30, min_periods=1).median())
    usual_day = pd.Series(usual_day.reindex(pd.MultiIndex.from_arrays([t.card_id, d])).to_numpy(), index=t.index)
    typical_day = spend[spend.index.get_level_values(1) < "2020-01-01"].median()   # the median card-day of 2019
    typical_amount = t.amount[d < "2020-01-01"].median()                           # the median payment of 2019
    spend_24h = last_24h(t.amount)
    return pd.DataFrame({
        "log_amount": np.log(t.amount),
        "hour": hour,
        "log_spend_24h_vs_usual_day": np.log1p(spend_24h) - np.log1p(usual_day.fillna(typical_day)),
        "night_payments_24h": last_24h(night),
        "amount_vs_usual": t.amount / median.fillna(typical_amount),
        "robust_z": (t.amount - median) / ((q3 - q1) / 1.349).replace(0, np.nan),   # NaN: fewer than 5 past payments
        "night": night,
        "spend_24h": spend_24h, "usual_day": usual_day,                      # kept for the failure case
    }, index=tx.index)

X = card_features(tx)
FEATURES = ["log_amount", "hour", "log_spend_24h_vs_usual_day", "night_payments_24h"]
print(X.groupby(tx.is_fraud)[FEATURES].median().rename(index={0: "honest", 1: "fraud"}).round(2).to_string())

> **Check.** The frauds have a much higher `log_spend_24h_vs_usual_day` (about 2.7 against 0.34) and more night payments in the last 24 hours (4 against 1).

## 4. Split by time

The detectors learn from 2019. You choose with the first half of 2020 (validation). You test once on July to November 2020. December 2020 is the new input.

In [ ]:
reference = tx[day < "2020-01-01"]                                  # 2019: learn normal, and train with labels
validation = tx[(day >= "2020-01-01") & (day < "2020-07-01")]      # choose the detector and its settings
test = tx[(day >= "2020-07-01") & (day < "2020-12-01")]            # used once, at the end
december = tx[day >= "2020-12-01"]                                  # the new input
for name, part in [("reference", reference), ("validation", validation), ("test", test), ("december", december)]:
    print(f"{name:>10}: {part.ts.min():%Y-%m-%d} to {part.ts.max():%Y-%m-%d}, {len(part):,} payments, "
          f"{part.is_fraud.sum():,} fraud ({part.is_fraud.mean():.2%})")

## 5. Baselines first

Five simple scores. Each day, the team reads the 20 highest scores. The rule gives 1 or 0, so its ties keep the time order; the ranked rule puts the payments most over the limit first.

In [ ]:
def baselines(part):
    x = X.loc[part.index]
    z = (part.amount - reference.amount.mean()) / reference.amount.std()
    rule = (x.night == 1) & (x.amount_vs_usual >= 3)                      # night, and 3 times the card's usual amount
    return [
        score_at_budget(part, np.zeros(len(part)), "no detector", "nothing is reviewed", budget=0),
        score_at_budget(part, z.abs(), "z-score", "|z| of the amount, one baseline for all"),
        score_at_budget(part, x.robust_z.abs().fillna(0), "robust z per card", "|amount - median| / (IQR / 1.349), the card's last 30 amounts"),
        score_at_budget(part, rule.astype(int), "rule", "night and 3 times the usual amount", threshold=1),
        score_at_budget(part, np.where(rule, x.amount_vs_usual, 0), "rule, ranked", "the same rule, most over first", threshold=0.5),
    ]

table = pd.DataFrame(baselines(validation))
table

> **Check.** On the validation months, the z-score of the amount has a precision of 0.288 at 20 a day and costs 14,071 dollars a week. Doing nothing costs 51,430.

## 6. Detectors without labels

Isolation Forest and the kNN distance learn "normal" from 2019 and never see a label.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

forest_raw = IsolationForest(n_estimators=200, random_state=0).fit(X.loc[reference.index, ["log_amount", "hour"]])
forest = IsolationForest(n_estimators=200, random_state=0).fit(X.loc[reference.index, FEATURES])
scaler = StandardScaler().fit(X.loc[reference.index, FEATURES])
sample = np.random.default_rng(0).choice(len(reference), 20_000, replace=False)
neighbours = NearestNeighbors(n_neighbors=10).fit(scaler.transform(X.loc[reference.index, FEATURES])[sample])

def unsupervised(part):
    x = X.loc[part.index]
    knn_distance = neighbours.kneighbors(scaler.transform(x[FEATURES]))[0][:, -1]   # distance to the 10th neighbour
    return [
        score_at_budget(part, -forest_raw.score_samples(x[["log_amount", "hour"]]), "Isolation Forest", "raw: amount and hour; 200 trees"),
        score_at_budget(part, -forest.score_samples(x[FEATURES]), "Isolation Forest", "card features; 200 trees"),
        score_at_budget(part, knn_distance, "kNN distance", "card features; k = 10, 20,000 reference payments"),
    ]

table = pd.concat([table, pd.DataFrame(unsupervised(validation))], ignore_index=True)
table

> **Check.** Isolation Forest on the card features: precision 0.37, but 16,203 dollars a week, more than the z-score. It finds more frauds, but smaller ones.

## 7. A supervised model, trained on the past

Gradient boosting learns from the labels of 2019 only, and scores the validation months.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

INPUTS = FEATURES + ["category"]

def model_inputs(part):
    x = X.loc[part.index, FEATURES].copy()
    x["category"] = part.category.astype(pd.CategoricalDtype(sorted(tx.category.unique())))
    return x

def train(part):
    model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, early_stopping=False,
                                           categorical_features="from_dtype", random_state=0)
    return model.fit(model_inputs(part), part.is_fraud)

model = train(reference)                       # labels of 2019 only
p_fraud = model.predict_proba(model_inputs(validation))[:, 1]
table = pd.concat([table, pd.DataFrame([
    score_at_budget(validation, p_fraud, "gradient boosting", "trained on 2019 labels; ranked by probability"),
    score_at_budget(validation, validation.is_fraud * (validation.amount + CHARGEBACK_FEE), "perfect", "knows the answers; largest first: the ceiling"),
])], ignore_index=True)
table

> **Check.** Gradient boosting: precision 0.506 and 9,960 dollars a week. The perfect row shows the ceiling at 20 a day.

### A threshold of 0.5 instead of the budget

In [ ]:
alerts = p_fraud >= 0.5
per_day = pd.Series(alerts).groupby(validation.ts.dt.floor("D").to_numpy()).sum()
print(f"threshold 0.5: {alerts.sum() / per_day.size:.1f} alerts a day (from {per_day.min()} to {per_day.max()}), "
      f"precision {validation.is_fraud[alerts].mean():.3f}, recall {alerts[validation.is_fraud.to_numpy() == 1].mean():.3f}")
print(f"days with more than {BUDGET} alerts: {(per_day > BUDGET).sum()} of {per_day.size}")
print(f"accuracy at 0.5: {(alerts == (validation.is_fraud == 1)).mean():.2%}")

A fixed threshold sends a different number of alerts each day: from 0 to 38. On 29 days it sends more than the team can read.

### Failure case: the new cards

The next cell repeats the model with one change: a card with no past is compared with **itself**, not with a typical card.

> **Predict.** Will the model with "compared with itself" do better or worse on the validation months?

In [ ]:
first_day = tx.groupby("card_id").ts.transform("min").dt.floor("D") == day     # the card's first day of payments
for when, part in [("2019", reference), ("validation", validation)]:
    f = first_day[part.index]
    print(f"{when}: {f.sum():,} payments on a card's first day, {part.is_fraud[f].sum()} of them fraud")
X_typical = X
X = X.assign(log_spend_24h_vs_usual_day=np.log1p(X.spend_24h) - np.log1p(X.usual_day.fillna(X.spend_24h)))  # compared with itself
own = score_at_budget(validation, train(reference).predict_proba(model_inputs(validation))[:, 1], "gradient boosting", "no history: compared with itself")
X = X_typical
print(pd.DataFrame([own, table.iloc[-2]]).to_string(index=False))

In 2019, a card's first day was almost always honest, because the data starts on 1 January 2019. In 2020, every card's first day was fraud. With "compared with itself", a new card's first payments look exactly usual, and the model learned to trust them. The lesson page explains the fix.

## 8. Rank by expected loss

The model gives a probability. The team should read the payments that are likely to **cost the most**: probability × (amount + 15).

In [ ]:
def expected_loss(part, p):
    """The money that a payment is likely to cost if nobody reviews it: P(fraud) x (amount + fee)."""
    return p * (part.amount.to_numpy() + CHARGEBACK_FEE)

table = pd.concat([table, pd.DataFrame([
    score_at_budget(validation, expected_loss(validation, p_fraud), "gradient boosting", "the same model; ranked by expected loss"),
])], ignore_index=True)
table.sort_values("cost per week ($)")

> **Check.** The same model, ranked by expected loss: 6,077 dollars a week, against 9,960 when ranked by probability. This is the choice.

## 9. The test, once

Now run every detector on July to November 2020, once. Do not change anything after this cell.

In [ ]:
p_test = model.predict_proba(model_inputs(test))[:, 1]
final = pd.DataFrame(baselines(test) + unsupervised(test) + [
    score_at_budget(test, p_test, "gradient boosting", "trained on 2019 labels; ranked by probability"),
    score_at_budget(test, expected_loss(test, p_test), "gradient boosting", "the same model; ranked by expected loss"),
    score_at_budget(test, test.is_fraud * (test.amount + CHARGEBACK_FEE), "perfect", "knows the answers; largest first: the ceiling"),
])
final.sort_values("cost per week ($)")

> **Check.** The chosen model costs 4,421 dollars a week on the test months, with a precision of 0.431 and a recall of 0.753.

## 10. What if the team could read more?

The cost curve of the chosen model on the validation months.

In [ ]:
loss_val = expected_loss(validation, p_fraud)
curve = pd.DataFrame([{"budget": b, **score_at_budget(validation, loss_val, "gradient boosting", budget=b)}
                      for b in (5, 10, 15, 20, 30, 40, 60, 80, 100)])
fig, ax = plt.subplots()
ax.plot(curve.budget, curve["cost per week ($)"], marker="o")
ax.axvline(BUDGET, color="grey", ls=":", label=f"the team's {BUDGET} a day")
ax.set_xlabel("alerts a day (the budget)"); ax.set_ylabel("cost per week ($)"); ax.legend()
plt.show()
curve[["budget", "precision at budget", "recall", "false alerts per week", "cost per week ($)"]]

> **Check.** The cost falls from 6,077 dollars a week at 20 alerts a day to 3,026 at 40, and then rises again.

## 11. December: the new input

Retrain on everything labelled up to 31 October (the November labels are not complete yet), and run on December 2020.

In [ ]:
labelled = tx[day < "2020-11-01"]                 # chargebacks take weeks: November's labels are not complete yet
model_dec = train(labelled)
p_dec = model_dec.predict_proba(model_inputs(december))[:, 1]
loss_dec = expected_loss(december, p_dec)
dec_table = pd.DataFrame([
    score_at_budget(december, np.zeros(len(december)), "no detector", "nothing is reviewed", budget=0),
    score_at_budget(december, (december.amount - reference.amount.mean()).abs(), "z-score", "|z| of the amount"),
    score_at_budget(december, -forest.score_samples(X.loc[december.index, FEATURES]), "Isolation Forest", "card features; fit on 2019"),
    score_at_budget(december, expected_loss(december, model.predict_proba(model_inputs(december))[:, 1]), "gradient boosting", "trained on 2019; expected loss"),
    score_at_budget(december, loss_dec, "gradient boosting", "retrained to 2020-10-31; expected loss"),
    score_at_budget(december, december.is_fraud * (december.amount + CHARGEBACK_FEE), "perfect", "knows the answers; largest first: the ceiling"),
])
print(dec_table.to_string(index=False))
per_day = december.groupby(december.ts.dt.floor("D")).is_fraud.agg(["size", "sum"])
print(f"December: {per_day['size'].mean():,.0f} payments a day (2020 before December: "
      f"{tx[(day >= '2020-01-01') & (day < '2020-12-01')].groupby(day).size().mean():,.0f}), {per_day['sum'].mean():.1f} frauds a day")

December has twice as many payments and fewer frauds. The base rate falls to 0.18%, so the precision at 20 a day falls for every detector, even the retrained one.

The queue of one day, 22 December 2020, with the reasons the team would read:

In [ ]:
today = december[december.ts.dt.floor("D") == "2020-12-22"]
queue = review(today, loss_dec[december.index.get_indexer(today.index)])
queue = queue[queue.reviewed].join(X[["amount_vs_usual", "night_payments_24h"]]).join(tx[["card_id", "category"]])
queue["p_fraud"] = p_dec[december.index.get_indexer(queue.index)]
queue = queue.rename(columns={"score": "expected_loss"}).sort_values("expected_loss", ascending=False)
print(f"2020-12-22: {len(today):,} payments, {today.is_fraud.sum()} frauds; the team's {BUDGET} alerts hold {queue.is_fraud.sum()}")
queue[["ts", "card_id", "category", "amount", "amount_vs_usual", "night_payments_24h", "p_fraud", "expected_loss", "is_fraud"]].head(8).round(2)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 12. Check yourself

**Your turn 1.** What is the precision of the chosen model (ranked by expected loss) at a budget of **10** alerts a day, on the validation months? Write it as `precision_at_10`.

In [ ]:
precision_at_10 = score_at_budget(validation, expected_loss(validation, p_fraud), "gradient boosting", budget=10)["precision at budget"]

In [ ]:
expect('precision_at_10', precision_at_10, 0.674)

**Your turn 2.** Change one thing in the rule: 5 times the usual amount instead of 3, still at night and ranked. Write its comparison-table row on the validation months as `rule_5x` (use `score_at_budget`, with `threshold=0.5` as in the baselines).

> **Predict.** Will the stricter rule find more frauds at 20 a day, or fewer?

In [ ]:
x = X.loc[validation.index]
rule_5x = score_at_budget(validation, np.where((x.night == 1) & (x.amount_vs_usual >= 5), x.amount_vs_usual, 0),
                          "rule, ranked", "night and 5 times the usual amount", threshold=0.5)
rule_5x

In [ ]:
expect('the precision of rule_5x', rule_5x['precision at budget'] if isinstance(rule_5x, dict) else None, 0.285)

**Your turn 3.** How many of December's 258 frauds does the team review at 20 alerts a day with the retrained model (`loss_dec`)? Write it as `december_frauds_caught`. Hint: `review()` marks the reviewed payments.

In [ ]:
december_frauds_caught = int(review(december, loss_dec).query("reviewed").is_fraud.sum())
december_frauds_caught

In [ ]:
expect('december_frauds_caught', december_frauds_caught, 174)

## Recap

- With fraud at 0.5%, accuracy says nothing: "nothing is fraud" is 99.40% right.
- Compare every detector at the team's budget, under one measure: here, the cost per week.
- Without labels, sorting by amount was cheaper than Isolation Forest and kNN. With labels, gradient boosting ranked by expected loss was the cheapest: 4,421 dollars a week on the test months.
- A card with no past needs a deliberate default. December's higher volume cut the base rate to 0.18%, and with it the precision.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Faults in machine or server metrics](https://learning.nextia-ai.com/courses/anomaly-detection/m10/faults-in-machine-or-server-metrics/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m10/card-fraud-with-a-review-budget/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The card payments are simulated (Sparkov, CC0 1.0) and have no real people.